# Network Intrusion Detection System

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import socket
import struct
import threading
import time
from collections import defaultdict

class NetworkIDS:
    def __init__(self, contamination=0.1):
        self.model = IsolationForest(contamination=contamination, random_state=42)
        self.scaler = StandardScaler()
        self.connection_stats = defaultdict(lambda: {'packets': 0, 'bytes': 0, 'last_seen': 0})
        self.trained = False

    def extract_features(self, packet_data):
        features = []
        for src_ip, dst_ip, size, timestamp in packet_data:
            key = f"{src_ip}:{dst_ip}"
            self.connection_stats[key]['packets'] += 1
            self.connection_stats[key]['bytes'] += size
            self.connection_stats[key]['last_seen'] = timestamp

            time_diff = timestamp - self.connection_stats[key].get('first_seen', timestamp)
            packet_rate = self.connection_stats[key]['packets'] / max(time_diff, 1)
            byte_rate = self.connection_stats[key]['bytes'] / max(time_diff, 1)

            features.append([size, packet_rate, byte_rate, len(src_ip.split('.')),
                           self.connection_stats[key]['packets']])
        return np.array(features)

    def train(self, normal_traffic):
        features = self.extract_features(normal_traffic)
        features_scaled = self.scaler.fit_transform(features)
        self.model.fit(features_scaled)
        self.trained = True

    def detect_anomalies(self, traffic_data):
        if not self.trained:
            raise ValueError("Model must be trained first")
        features = self.extract_features(traffic_data)
        features_scaled = self.scaler.transform(features)
        predictions = self.model.predict(features_scaled)
        return predictions == -1

    def generate_sample_data(self, n_samples=1000, anomaly_ratio=0.05):
        normal_data = []
        for i in range(int(n_samples * (1 - anomaly_ratio))):
            src_ip = f"192.168.1.{np.random.randint(1, 255)}"
            dst_ip = f"10.0.0.{np.random.randint(1, 255)}"
            size = np.random.normal(512, 100)
            timestamp = time.time() + i
            normal_data.append((src_ip, dst_ip, max(64, int(size)), timestamp))

        anomaly_data = []
        for i in range(int(n_samples * anomaly_ratio)):
            src_ip = f"192.168.1.{np.random.randint(1, 255)}"
            dst_ip = f"10.0.0.{np.random.randint(1, 255)}"
            size = np.random.exponential(5000)
            timestamp = time.time() + len(normal_data) + i
            anomaly_data.append((src_ip, dst_ip, int(size), timestamp))

        return normal_data, anomaly_data